# 01: Leitura de Volumes 3D e Metadados Auxiliares em PET/CT

**Objetivo:** Aprender na prática como carregar arquivos volumétricos no formato NIfTI (`.nii.gz`), inspecionar seus metadados espaciais físicos com `SimpleITK`, converter para arrays `NumPy` e integrar informações clínicas de planilhas Excel (`PT-details.xlsx` e `CT-details.xlsx`) e JSON (`label.json`).

---

## 1. Importações e Configuração de Caminhos

In [1]:
import sys
from pathlib import Path

# Adiciona o diretório raiz ao PYTHONPATH para permitir importar config e src
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

import config
from src import io_utils

print("Módulos importados com sucesso!")
print("Caminho base do dataset configurado em config.py:", config.RAIZ)

Módulos importados com sucesso!
Caminho base do dataset configurado em config.py: d:\usuarios\estela.ribeiro\Meus Documentos\Datasets\ENHANCE-PET_1.6k


## 2. Carregando os Arquivos de um Paciente

Vamos selecionar o paciente padrão `'0001'` e inspecionar os caminhos dos arquivos NIfTI do CT, PET e Máscara Cardíaca.

In [2]:
patient_id = "0001"
caminhos = config.get_caminhos(patient_id)

for modalidade, path in caminhos.items():
    status = "✓ Encontrado" if path.exists() else "✗ Não encontrado"
    print(f"{modalidade:18s}: {status} -> {path}")

CT                : ✓ Encontrado -> d:\usuarios\estela.ribeiro\Meus Documentos\Datasets\ENHANCE-PET_1.6k\images\CT_0001-1000\0001.nii.gz
PET               : ✓ Encontrado -> d:\usuarios\estela.ribeiro\Meus Documentos\Datasets\ENHANCE-PET_1.6k\images\PT\0001.nii.gz
Mascara_Cardiaca  : ✓ Encontrado -> d:\usuarios\estela.ribeiro\Meus Documentos\Datasets\ENHANCE-PET_1.6k\ground-truth\Cardiac\0001.nii.gz


## 3. Leitura e Inspeção de Metadados com SimpleITK

O SimpleITK preserva a geometria física do paciente. Vamos carregar o CT e o PET e extrair:
- **Size:** número de voxels $(X, Y, Z)$;
- **Spacing:** tamanho de cada voxel em milímetros $(s_x, s_y, s_z)$;
- **Origin:** coordenada física do primeiro voxel $(x_0, y_0, z_0)$;
- **Direction:** matriz de orientação 3x3.

In [3]:
if caminhos["CT"].exists() and caminhos["PET"].exists():
    ct_img = io_utils.load_sitk_image(caminhos["CT"])
    pet_img = io_utils.load_sitk_image(caminhos["PET"])
    
    io_utils.print_volume_summary("CT 0001", ct_img)
    io_utils.print_volume_summary("PET Original 0001", pet_img)
else:
    print("Arquivos de dados não disponíveis no caminho local. Verifique config.py.")


  Volume: CT 0001
  Dimensões (voxels): (512, 512, 365)  →  (x, y, z)
  Array shape (numpy): (365, 512, 512)    →  (z, y, x)
  Spacing (mm/voxel): ('0.8633', '0.8633', '2.5000')
  Origem (mm):        ('-220.57', '21.57', '-1155.50')
  Tipo do pixel:      32-bit float
  Intensidade mín:    -1024.0000
  Intensidade máx:    3071.0000
  Intensidade média:  -625.2960

  Volume: PET Original 0001
  Dimensões (voxels): (400, 400, 304)  →  (x, y, z)
  Array shape (numpy): (304, 400, 400)    →  (z, y, x)
  Spacing (mm/voxel): ('2.0364', '2.0364', '3.0000')
  Origem (mm):        ('-406.69', '205.44', '-1154.50')
  Tipo do pixel:      32-bit float
  Intensidade mín:    0.0000
  Intensidade máx:    149189.7656
  Intensidade média:  228.5280


## 4. Conversão para NumPy: A Ordem dos Eixos

> **Dica Didática:**
> - SimpleITK indexa em $(X, Y, Z) = (\text{largura}, \text{altura}, \text{profundidade})$;
> - NumPy indexa em $(Z, Y, X) = (\text{fatia}, \text{linha}, \text{coluna})$.

A função `io_utils.sitk_to_numpy()` já entrega o array na convenção canônica do NumPy `(z, y, x)`.

In [4]:
if caminhos["CT"].exists():
    ct_arr = io_utils.sitk_to_numpy(ct_img)
    print("Shape do array NumPy do CT (Z, Y, X):", ct_arr.shape)
    print(f"Faixa de intensidade em HU: [{ct_arr.min():.1f}, {ct_arr.max():.1f}]")

Shape do array NumPy do CT (Z, Y, X): (365, 512, 512)
Faixa de intensidade em HU: [-1024.0, 3071.0]


## 5. Metadados Clínicos Tabulares (PT-details.xlsx e CT-details.xlsx)

Vamos ler a planilha de aquisição PET e extrair o peso do paciente e a dose administrada para o cálculo de SUV.

In [5]:
if config.XLSX_PT_PATH.exists():
    df_pt = io_utils.load_pt_details(config.XLSX_PT_PATH)
    meta_pt = io_utils.get_patient_metadata(patient_id, df_pt)
    print(f"Metadados do Paciente {patient_id} em PT-details:")
    for k, v in meta_pt.items():
        print(f"  {k:30s}: {v}")
else:
    print(f"Planilha PT-details não encontrada em {config.XLSX_PT_PATH}")

Metadados do Paciente 0001 em PT-details:
  Patient                       : 1
  Data-Source                   : AutoPET Challenge
  Diagnosis                     : NEGATIVE
  Sex                           : F
  Age                           : 61.0
  Weight [kg]                   : 91.0
  Height [m]                    : 1.75
  Dose [MBq]                    : 305.0
  Acquisition Date              : 2005-08-12 00:00:00
  Acquisition Time              : 12:14:51
  Radiopharmaceutical Start Time: 11:12:00
  Series Time                   : 12:12:35
  Units                         : BQML
  Slice Thickness               : 3.0
  Slope                         : 1.17561
  Intercept                     : 0
  Manufacturer                  : SIEMENS
  Model                         : Biograph128_mCT


## 6. Mapeamento de Rótulos Anatômicos (label.json)

Inspecionamos os nomes anatômicos das estruturas cardíacas segmentadas.

In [7]:
labels = config.ROTULOS_CARDIACOS
print("Rótulos Anatômicos Cardíacos mapeados no projeto:")
for label_id, nome in labels.items():
    destaque = "-> FOCO PRINCIPAL" if label_id == 1 else ""
    print(f"  Label {label_id:2d} -> {nome:25s} {destaque}")

Rótulos Anatômicos Cardíacos mapeados no projeto:
  Label  0 -> background                
  Label  1 -> heart_myocardium          -> FOCO PRINCIPAL
  Label  2 -> heart_atrium_left         
  Label  3 -> heart_atrium_right        
  Label  4 -> heart_ventricle_left      
  Label  5 -> heart_ventricle_right     
  Label  6 -> aorta                     
  Label  7 -> iliac_artery_left         
  Label  8 -> iliac_artery_right        
  Label  9 -> iliac_vena_left           
  Label 10 -> iliac_vena_right          
  Label 11 -> inferior_vena_cava        
  Label 12 -> portal_splenic_vein       
  Label 13 -> pulmonary_artery          
